# Imports/Dependencies

In [2]:
# Cell 1 — Install
!pip install torch torchvision transformers==4.41.2 safetensors matplotlib -q
!pip install git+https://github.com/openai/CLIP.git -q
# Cell 2 — Clone HF model repo
!git clone https://huggingface.co/lorebianchi98/Talk2DINO-ViTB /content/Talk2DINO-ViTB
!pip install 'git+https://github.com/facebookresearch/detectron2.git' -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.8/43.8 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 119.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 45.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 93.8 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 4.7 MB/s eta 0:00:00


# Test on 1 Image (Not Needed Now)

In [52]:
# import sys, os

# pkg_dir = "/content/Talk2DINO-ViTB"
# init_path = os.path.join(pkg_dir, "__init__.py")
# if not os.path.exists(init_path):
#     with open(init_path, "w") as f:
#         f.write("")

# if not os.path.exists("/content/talk2dino_vitb"):
#     os.symlink(pkg_dir, "/content/talk2dino_vitb")

# sys.path.insert(0, "/content")

# import torch
# import numpy as np
# import matplotlib.pyplot as plt
# from PIL import Image as PILImage
# from torchvision.io import read_image
# from safetensors.torch import load_file

# from talk2dino_vitb.configuration_talk2dino import Talk2DINOConfig
# from talk2dino_vitb.modeling_talk2dino import Talk2DINO

# device = 'cuda' if torch.cuda.is_available() else 'cpu'

# config = Talk2DINOConfig.from_pretrained("/content/Talk2DINO-ViTB")
# model = Talk2DINO(config)

# model = model.to_empty(device=device)
# state_dict = load_file("/content/Talk2DINO-ViTB/model.safetensors", device=device)
# model.load_state_dict(state_dict, strict=False, assign=True)
# model.eval()

# from torchvision import transforms

# image_path = "/content/Screenshot_10-4-2026_202636_arxiv.org.jpeg"

# img_pil = PILImage.open(image_path).convert("RGB")
# image = transforms.ToTensor()(img_pil).to(device)  # [3, H, W] float
# image = (image * 255).to(torch.uint8)  # convert to uint8 like read_image does

# # categories = ["impervious surface", "building", "low vegetation", "tree","car"]
# categories = ["building-flooded","building-non-flooded","road-flooded", "road-non-flooded","water","tree","vehicle","pool","grass"]

# with torch.no_grad():
#     image_embed = model.encode_image(image)
#     image_embed = image_embed / image_embed.norm(dim=-1, keepdim=True)

#     text_embeds = []
#     for cat in categories:
#         t = model.encode_text(cat)
#         t = t / t.norm(dim=-1, keepdim=True)
#         text_embeds.append(t)
#     text_embeds = torch.cat(text_embeds, dim=0)

# similarity = (image_embed @ text_embeds.T)
# seg_map = similarity[0].argmax(dim=-1).cpu().numpy()

# h = w = 37
# seg_map = seg_map.reshape(h, w)

# # Upsample to original size
# original = np.array(PILImage.open(image_path).convert("RGB"))
# seg_pil = PILImage.fromarray(seg_map.astype(np.uint8)).resize(
#     (original.shape[1], original.shape[0]), PILImage.NEAREST
# )
# seg_array = np.array(seg_pil)

# # Create colored overlay
# cmap = plt.cm.tab10
# seg_colored = (cmap(seg_array / max(seg_array.max(), 1))[:, :, :3] * 255).astype(np.uint8)
# alpha = 0.35
# overlay = (original * (1 - alpha) + seg_colored * alpha).astype(np.uint8)

# # Plot
# fig, axes = plt.subplots(1, 2, figsize=(14, 6))
# axes[0].imshow(original)
# axes[0].set_title("Input")
# axes[0].axis("off")

# axes[1].imshow(overlay)
# axes[1].set_title("Segmentation Overlay")
# axes[1].axis("off")

# for i, cat in enumerate(categories):
#     color = cmap(i / max(len(categories) - 1, 1))[:3]
#     axes[1].plot([], [], 's', color=color, markersize=10, label=cat)
# axes[1].legend(loc='upper left', fontsize=8, framealpha=0.8)

# plt.tight_layout()
# plt.show()

In [34]:
# from google.colab import drive
# drive.mount('/content/drive')

Mounted at /content/drive


# Load Datasets & Model

In [ ]:
import os, sys, torch, numpy as np
from PIL import Image
from torchvision import transforms
from detectron2.data import DatasetCatalog, MetadataCatalog
from detectron2.data.datasets import load_sem_seg
from detectron2.evaluation import SemSegEvaluator

os.environ["DETECTRON2_DATASETS"] = "/content/drive/MyDrive/OVRSISS_test"
DETECTRON2_DATASETS = os.getenv("DETECTRON2_DATASETS")

# Load Talk2DINO (once)
sys.path.insert(0, "/content")
from talk2dino_vitb.configuration_talk2dino import Talk2DINOConfig
from talk2dino_vitb.modeling_talk2dino import Talk2DINO
from safetensors.torch import load_file

device = "cuda" if torch.cuda.is_available() else "cpu"
t2d_config = Talk2DINOConfig.from_pretrained("/content/Talk2DINO-ViTB")
model = Talk2DINO(t2d_config)
model = model.to_empty(device=device)
state_dict = load_file("/content/Talk2DINO-ViTB/model.safetensors", device=device)
model.load_state_dict(state_dict, strict=False, assign=True)
model.eval()

# All dataset configs
DATASET_CONFIGS = {
    "potsdam_all": {
        "classes": ["impervious surface", "building", "low vegetation", "tree", "car", "clutter"],
        "test_classes": ["impervious surface", "building", "low vegetation", "tree", "car"],
        "image_dir": os.path.join(DETECTRON2_DATASETS, "PotsdamSplit", "img_dir", "val"),
        "gt_dir": os.path.join(DETECTRON2_DATASETS, "PotsdamSplit", "ann_dir", "val"),
        "gt_ext": "png", "image_ext": "png", "ignore_label": 5,
    },
    "FloodNet": {
        "classes": ["Background", "building-flooded", "building-non-flooded", "road-flooded",
                     "road-non-flooded", "water", "tree", "vehicle", "pool", "grass"],
        "test_classes": ["Background", "building-flooded", "building-non-flooded", "road-flooded",
                          "road-non-flooded", "water", "tree", "vehicle", "pool", "grass"],
        "image_dir": os.path.join(DETECTRON2_DATASETS, "FloodNet", "val+test", "img"),
        "gt_dir": os.path.join(DETECTRON2_DATASETS, "FloodNet", "val+test", "lbl"),
        "gt_ext": "png", "image_ext": "jpg", "ignore_label": 0,
    },
    "FLAIR_test": {
        "classes": ["building", "pervious surface", "impervious surface", "bare soil",
                     "water", "coniferous", "deciduous", "brushwood", "vineyard",
                     "herbaceous vegetation", "agricultural land", "plowed land", "other"],
        "test_classes": ["building", "pervious-surface", "impervious-surface", "bare soil",
                          "water", "coniferous", "deciduous", "brushwood", "vineyard",
                          "herbaceous vegetation", "agricultural land", "plowed land"],
        "image_dir": os.path.join(DETECTRON2_DATASETS, "FLAIR_test", "image"),
        "gt_dir": os.path.join(DETECTRON2_DATASETS, "FLAIR_test", "mask"),
        "gt_ext": "png", "image_ext": "png", "ignore_label": 12,
    },
    "FAST_val": {
        "classes": ["A220", "A321", "A330", "A350", "ARJ21", "Baseball-Field", "Basketball-Court",
                     "Boeing737", "Boeing747", "Boeing777", "Boeing787", "Bridge", "Bus", "C919",
                     "Cargo-Truck", "Dry-Cargo-Ship", "Dump-Truck", "Engineering-Ship", "Excavator",
                     "Fishing-Boat", "Football-Field", "Intersection", "Liquid-Cargo-Ship", "Motorboat",
                     "other-airplane", "other-ship", "other-vehicle", "Passenger-Ship", "Roundabout",
                     "Small-Car", "Tennis-Court", "Tractor", "Trailer", "Truck-Tractor", "Tugboat",
                     "Van", "Warship"],
        "test_classes": ["A220", "A321", "A330", "A350", "ARJ21", "Baseball-Field", "Basketball-Court",
                          "Boeing737", "Boeing747", "Boeing777", "Boeing787", "Bridge", "Bus", "C919",
                          "Cargo-Truck", "Dry-Cargo-Ship", "Dump-Truck", "Engineering-Ship", "Excavator",
                          "Fishing-Boat", "Football-Field", "Intersection", "Liquid-Cargo-Ship", "Motorboat",
                          "other-airplane", "other-ship", "other-vehicle", "Passenger-Ship", "Roundabout",
                          "Small-Car", "Tennis-Court", "Tractor", "Trailer", "Truck-Tractor", "Tugboat",
                          "Van", "Warship"],
        "image_dir": os.path.join(DETECTRON2_DATASETS, "FAST", "val", "images"),
        "gt_dir": os.path.join(DETECTRON2_DATASETS, "FAST", "val", "semlabels", "gray"),
        "gt_ext": "png", "image_ext": "png", "ignore_label": 255,
    },
}

print("Setup done. Model loaded.")

# 1. Potsdam

In [58]:
EVAL_DATASET = "potsdam_all"  # <--- CHANGE THIS
MAX_IMAGES = None  # Set to 50 for quick test, None for full run

cfg = DATASET_CONFIGS[EVAL_DATASET]

if EVAL_DATASET in DatasetCatalog:
    DatasetCatalog.remove(EVAL_DATASET)
    MetadataCatalog.remove(EVAL_DATASET)

DatasetCatalog.register(EVAL_DATASET,
    lambda: load_sem_seg(cfg["gt_dir"], cfg["image_dir"], gt_ext=cfg["gt_ext"], image_ext=cfg["image_ext"]))
MetadataCatalog.get(EVAL_DATASET).set(
    stuff_classes=cfg["classes"], image_root=cfg["image_dir"],
    seg_seg_root=cfg["gt_dir"], evaluator_type="sem_seg", ignore_label=cfg["ignore_label"])

dataset_dicts = DatasetCatalog.get(EVAL_DATASET)

with torch.no_grad():
    text_embeds = torch.cat([model.encode_text(c) / model.encode_text(c).norm(dim=-1, keepdim=True) for c in cfg["test_classes"]], dim=0)

num_classes = len(cfg["test_classes"])
evaluator = SemSegEvaluator(EVAL_DATASET, distributed=False, output_dir=f"./eval_output/{EVAL_DATASET}")
evaluator.reset()

n = min(len(dataset_dicts), MAX_IMAGES) if MAX_IMAGES else len(dataset_dicts)
for i, entry in enumerate(dataset_dicts):
    if i >= n:
        break
    img_pil = Image.open(entry["file_name"]).convert("RGB")
    image = (transforms.ToTensor()(img_pil) * 255).to(torch.uint8).to(device)
    h, w = img_pil.size[1], img_pil.size[0]

    with torch.no_grad():
        image_embed = model.encode_image(image)
        image_embed = image_embed / image_embed.norm(dim=-1, keepdim=True)

    seg_logits = (image_embed @ text_embeds.T)[0]
    ps = int(round(np.sqrt(seg_logits.shape[0])))
    seg_logits = torch.nn.functional.interpolate(
        seg_logits.reshape(ps, ps, num_classes).permute(2,0,1).unsqueeze(0).float(),
        size=(h,w), mode="bilinear", align_corners=False)[0]

    evaluator.process([{"file_name": entry["file_name"]}], [{"sem_seg": seg_logits.cpu()}])
    if (i+1) % 50 == 0 or (i+1) == n:
        print(f"  [{i+1}/{n}]")

results = evaluator.evaluate()
print(f"\n{'='*60}\n  Talk2DINO on {EVAL_DATASET} ({n} images)\n{'='*60}")
for k, v in results["sem_seg"].items():
    print(f"  {k}: {v:.4f}" if isinstance(v, float) else f"  {k}: {v}")

  [50/50]

  Talk2DINO on potsdam_all (50 images)
  mIoU: 45.8352
  fwIoU: 52.5386
  IoU-impervious surface: 38.0924
  BoundaryIoU-impervious surface: 62.4503
  min(IoU, B-Iou)-impervious surface: 38.0924
  IoU-building: 65.9122
  BoundaryIoU-building: 16.2643
  min(IoU, B-Iou)-building: 16.2643
  IoU-low vegetation: 56.1094
  BoundaryIoU-low vegetation: 24.2670
  min(IoU, B-Iou)-low vegetation: 24.2670
  IoU-tree: 33.5127
  BoundaryIoU-tree: 12.1856
  min(IoU, B-Iou)-tree: 12.1856
  IoU-car: 35.5492
  BoundaryIoU-car: 11.9017
  min(IoU, B-Iou)-car: 11.9017
  IoU-clutter: nan
  BoundaryIoU-clutter: 0.0000
  min(IoU, B-Iou)-clutter: nan
  mACC: 69.9046
  pACC: 66.7293
  ACC-impervious surface: 80.7952
  ACC-building: 74.0671
  ACC-low vegetation: 61.9150
  ACC-tree: 42.6904
  ACC-car: 90.0553
  ACC-clutter: nan


# 2. FloodNet

In [57]:
EVAL_DATASET = "FloodNet"  # <--- CHANGE THIS
MAX_IMAGES = None  # Set to 50 for quick test, None for full run

cfg = DATASET_CONFIGS[EVAL_DATASET]

if EVAL_DATASET in DatasetCatalog:
    DatasetCatalog.remove(EVAL_DATASET)
    MetadataCatalog.remove(EVAL_DATASET)

DatasetCatalog.register(EVAL_DATASET,
    lambda: load_sem_seg(cfg["gt_dir"], cfg["image_dir"], gt_ext=cfg["gt_ext"], image_ext=cfg["image_ext"]))
MetadataCatalog.get(EVAL_DATASET).set(
    stuff_classes=cfg["classes"], image_root=cfg["image_dir"],
    seg_seg_root=cfg["gt_dir"], evaluator_type="sem_seg", ignore_label=cfg["ignore_label"])

dataset_dicts = DatasetCatalog.get(EVAL_DATASET)

with torch.no_grad():
    text_embeds = torch.cat([model.encode_text(c) / model.encode_text(c).norm(dim=-1, keepdim=True) for c in cfg["test_classes"]], dim=0)

num_classes = len(cfg["test_classes"])
evaluator = SemSegEvaluator(EVAL_DATASET, distributed=False, output_dir=f"./eval_output/{EVAL_DATASET}")
evaluator.reset()

n = min(len(dataset_dicts), MAX_IMAGES) if MAX_IMAGES else len(dataset_dicts)
for i, entry in enumerate(dataset_dicts):
    if i >= n:
        break
    img_pil = Image.open(entry["file_name"]).convert("RGB")
    image = (transforms.ToTensor()(img_pil) * 255).to(torch.uint8).to(device)
    h, w = img_pil.size[1], img_pil.size[0]

    with torch.no_grad():
        image_embed = model.encode_image(image)
        image_embed = image_embed / image_embed.norm(dim=-1, keepdim=True)

    seg_logits = (image_embed @ text_embeds.T)[0]
    ps = int(round(np.sqrt(seg_logits.shape[0])))
    seg_logits = torch.nn.functional.interpolate(
        seg_logits.reshape(ps, ps, num_classes).permute(2,0,1).unsqueeze(0).float(),
        size=(h,w), mode="bilinear", align_corners=False)[0]

    evaluator.process([{"file_name": entry["file_name"]}], [{"sem_seg": seg_logits.cpu()}])
    if (i+1) % 50 == 0 or (i+1) == n:
        print(f"  [{i+1}/{n}]")

results = evaluator.evaluate()
print(f"\n{'='*60}\n  Talk2DINO on {EVAL_DATASET} ({n} images)\n{'='*60}")
for k, v in results["sem_seg"].items():
    print(f"  {k}: {v:.4f}" if isinstance(v, float) else f"  {k}: {v}")

Dataset: FloodNet
Samples: 898
Classes: ['Background', 'building-flooded', 'building-non-flooded', 'road-flooded', 'road-non-flooded', 'water', 'tree', 'vehicle', 'pool', 'grass']
Ignore label: 0


Using cache found in /root/.cache/torch/hub/facebookresearch_dinov2_main


Talk2DINO loaded. Text embeds: torch.Size([10, 768])
  [50/898]

  SemSegEvaluator Results — Talk2DINO on FloodNet
  mIoU: 40.3746
  fwIoU: 44.0468
  IoU-Background: nan
  BoundaryIoU-Background: 57.8157
  min(IoU, B-Iou)-Background: nan
  IoU-building-flooded: nan
  BoundaryIoU-building-flooded: 0.5249
  min(IoU, B-Iou)-building-flooded: nan
  IoU-building-non-flooded: 59.8232
  BoundaryIoU-building-non-flooded: 17.6721
  min(IoU, B-Iou)-building-non-flooded: 17.6721
  IoU-road-flooded: nan
  BoundaryIoU-road-flooded: 7.1488
  min(IoU, B-Iou)-road-flooded: nan
  IoU-road-non-flooded: 44.1145
  BoundaryIoU-road-non-flooded: 11.8333
  min(IoU, B-Iou)-road-non-flooded: 11.8333
  IoU-water: 0.0074
  BoundaryIoU-water: 6.0241
  min(IoU, B-Iou)-water: 0.0074
  IoU-tree: 50.9851
  BoundaryIoU-tree: 3.2647
  min(IoU, B-Iou)-tree: 3.2647
  IoU-vehicle: 22.2560
  BoundaryIoU-vehicle: 5.9283
  min(IoU, B-Iou)-vehicle: 5.9283
  IoU-pool: 52.8268
  BoundaryIoU-pool: 1.3240
  min(IoU, B-Iou)-pool: 

# 3. FLAIR

In [55]:
EVAL_DATASET = "FLAIR_test"  # <--- CHANGE THIS
MAX_IMAGES = None  # Set to 50 for quick test, None for full run

cfg = DATASET_CONFIGS[EVAL_DATASET]

if EVAL_DATASET in DatasetCatalog:
    DatasetCatalog.remove(EVAL_DATASET)
    MetadataCatalog.remove(EVAL_DATASET)

DatasetCatalog.register(EVAL_DATASET,
    lambda: load_sem_seg(cfg["gt_dir"], cfg["image_dir"], gt_ext=cfg["gt_ext"], image_ext=cfg["image_ext"]))
MetadataCatalog.get(EVAL_DATASET).set(
    stuff_classes=cfg["classes"], image_root=cfg["image_dir"],
    seg_seg_root=cfg["gt_dir"], evaluator_type="sem_seg", ignore_label=cfg["ignore_label"])

dataset_dicts = DatasetCatalog.get(EVAL_DATASET)

with torch.no_grad():
    text_embeds = torch.cat([model.encode_text(c) / model.encode_text(c).norm(dim=-1, keepdim=True) for c in cfg["test_classes"]], dim=0)

num_classes = len(cfg["test_classes"])
evaluator = SemSegEvaluator(EVAL_DATASET, distributed=False, output_dir=f"./eval_output/{EVAL_DATASET}")
evaluator.reset()

n = min(len(dataset_dicts), MAX_IMAGES) if MAX_IMAGES else len(dataset_dicts)
for i, entry in enumerate(dataset_dicts):
    if i >= n:
        break
    img_pil = Image.open(entry["file_name"]).convert("RGB")
    image = (transforms.ToTensor()(img_pil) * 255).to(torch.uint8).to(device)
    h, w = img_pil.size[1], img_pil.size[0]

    with torch.no_grad():
        image_embed = model.encode_image(image)
        image_embed = image_embed / image_embed.norm(dim=-1, keepdim=True)

    seg_logits = (image_embed @ text_embeds.T)[0]
    ps = int(round(np.sqrt(seg_logits.shape[0])))
    seg_logits = torch.nn.functional.interpolate(
        seg_logits.reshape(ps, ps, num_classes).permute(2,0,1).unsqueeze(0).float(),
        size=(h,w), mode="bilinear", align_corners=False)[0]

    evaluator.process([{"file_name": entry["file_name"]}], [{"sem_seg": seg_logits.cpu()}])
    if (i+1) % 50 == 0 or (i+1) == n:
        print(f"  [{i+1}/{n}]")

results = evaluator.evaluate()
print(f"\n{'='*60}\n  Talk2DINO on {EVAL_DATASET} ({n} images)\n{'='*60}")
for k, v in results["sem_seg"].items():
    print(f"  {k}: {v:.4f}" if isinstance(v, float) else f"  {k}: {v}")

Dataset: FLAIR_test
Samples: 15700
Classes: ['building', 'pervious-surface', 'impervious-surface', 'bare soil', 'water', 'coniferous', 'deciduous', 'brushwood', 'vineyard', 'herbaceous vegetation', 'agricultural land', 'plowed land']
Ignore label: 12


Using cache found in /root/.cache/torch/hub/facebookresearch_dinov2_main


Talk2DINO loaded. Text embeds: torch.Size([12, 768])
  [50/15700]

  SemSegEvaluator Results — Talk2DINO on FLAIR_test
  mIoU: 10.6913
  fwIoU: 17.9678
  IoU-building: 67.2691
  BoundaryIoU-building: 51.7002
  min(IoU, B-Iou)-building: 51.7002
  IoU-pervious surface: 0.8935
  BoundaryIoU-pervious surface: 1.2489
  min(IoU, B-Iou)-pervious surface: 0.8935
  IoU-impervious surface: 1.9111
  BoundaryIoU-impervious surface: 2.0064
  min(IoU, B-Iou)-impervious surface: 1.9111
  IoU-bare soil: nan
  BoundaryIoU-bare soil: 0.3561
  min(IoU, B-Iou)-bare soil: nan
  IoU-water: nan
  BoundaryIoU-water: 0.7013
  min(IoU, B-Iou)-water: nan
  IoU-coniferous: 0.4826
  BoundaryIoU-coniferous: 0.2329
  min(IoU, B-Iou)-coniferous: 0.2329
  IoU-deciduous: 0.4341
  BoundaryIoU-deciduous: 0.8007
  min(IoU, B-Iou)-deciduous: 0.4341
  IoU-brushwood: 0.0013
  BoundaryIoU-brushwood: 4.3046
  min(IoU, B-Iou)-brushwood: 0.0013
  IoU-vineyard: nan
  BoundaryIoU-vineyard: 0.4276
  min(IoU, B-Iou)-vineyard: nan
  

# 4. FAST

In [54]:
EVAL_DATASET = "FAST_val"  # <--- CHANGE THIS
MAX_IMAGES = None  # Set to 50 for quick test, None for full run

cfg = DATASET_CONFIGS[EVAL_DATASET]

if EVAL_DATASET in DatasetCatalog:
    DatasetCatalog.remove(EVAL_DATASET)
    MetadataCatalog.remove(EVAL_DATASET)

DatasetCatalog.register(EVAL_DATASET,
    lambda: load_sem_seg(cfg["gt_dir"], cfg["image_dir"], gt_ext=cfg["gt_ext"], image_ext=cfg["image_ext"]))
MetadataCatalog.get(EVAL_DATASET).set(
    stuff_classes=cfg["classes"], image_root=cfg["image_dir"],
    seg_seg_root=cfg["gt_dir"], evaluator_type="sem_seg", ignore_label=cfg["ignore_label"])

dataset_dicts = DatasetCatalog.get(EVAL_DATASET)

with torch.no_grad():
    text_embeds = torch.cat([model.encode_text(c) / model.encode_text(c).norm(dim=-1, keepdim=True) for c in cfg["test_classes"]], dim=0)

num_classes = len(cfg["test_classes"])
evaluator = SemSegEvaluator(EVAL_DATASET, distributed=False, output_dir=f"./eval_output/{EVAL_DATASET}")
evaluator.reset()

n = min(len(dataset_dicts), MAX_IMAGES) if MAX_IMAGES else len(dataset_dicts)
for i, entry in enumerate(dataset_dicts):
    if i >= n:
        break
    img_pil = Image.open(entry["file_name"]).convert("RGB")
    image = (transforms.ToTensor()(img_pil) * 255).to(torch.uint8).to(device)
    h, w = img_pil.size[1], img_pil.size[0]

    with torch.no_grad():
        image_embed = model.encode_image(image)
        image_embed = image_embed / image_embed.norm(dim=-1, keepdim=True)

    seg_logits = (image_embed @ text_embeds.T)[0]
    ps = int(round(np.sqrt(seg_logits.shape[0])))
    seg_logits = torch.nn.functional.interpolate(
        seg_logits.reshape(ps, ps, num_classes).permute(2,0,1).unsqueeze(0).float(),
        size=(h,w), mode="bilinear", align_corners=False)[0]

    evaluator.process([{"file_name": entry["file_name"]}], [{"sem_seg": seg_logits.cpu()}])
    if (i+1) % 50 == 0 or (i+1) == n:
        print(f"  [{i+1}/{n}]")

results = evaluator.evaluate()
print(f"\n{'='*60}\n  Talk2DINO on {EVAL_DATASET} ({n} images)\n{'='*60}")
for k, v in results["sem_seg"].items():
    print(f"  {k}: {v:.4f}" if isinstance(v, float) else f"  {k}: {v}")

Dataset: FAST_val
Samples: 3207
Classes: ['A220', 'A321', 'A330', 'A350', 'ARJ21', 'Baseball-Field', 'Basketball-Court', 'Boeing737', 'Boeing747', 'Boeing777', 'Boeing787', 'Bridge', 'Bus', 'C919', 'Cargo-Truck', 'Dry-Cargo-Ship', 'Dump-Truck', 'Engineering-Ship', 'Excavator', 'Fishing-Boat', 'Football-Field', 'Intersection', 'Liquid-Cargo-Ship', 'Motorboat', 'other-airplane', 'other-ship', 'other-vehicle', 'Passenger-Ship', 'Roundabout', 'Small-Car', 'Tennis-Court', 'Tractor', 'Trailer', 'Truck-Tractor', 'Tugboat', 'Van', 'Warship']
Ignore label: 255


Using cache found in /root/.cache/torch/hub/facebookresearch_dinov2_main


Talk2DINO loaded. Text embeds: torch.Size([37, 768])
  [50/3207]

  SemSegEvaluator Results — Talk2DINO on FAST_val
  mIoU: 10.9212
  fwIoU: 51.0568
  IoU-A220: 0.0000
  BoundaryIoU-A220: 64.6890
  min(IoU, B-Iou)-A220: 0.0000
  IoU-A321: 0.0000
  BoundaryIoU-A321: 0.0021
  min(IoU, B-Iou)-A321: 0.0000
  IoU-A330: 0.0000
  BoundaryIoU-A330: 1.1428
  min(IoU, B-Iou)-A330: 0.0000
  IoU-A350: 0.0000
  BoundaryIoU-A350: 0.1118
  min(IoU, B-Iou)-A350: 0.0000
  IoU-ARJ21: nan
  BoundaryIoU-ARJ21: 0.1102
  min(IoU, B-Iou)-ARJ21: nan
  IoU-Baseball-Field: nan
  BoundaryIoU-Baseball-Field: 0.0381
  min(IoU, B-Iou)-Baseball-Field: nan
  IoU-Basketball-Court: nan
  BoundaryIoU-Basketball-Court: 0.0081
  min(IoU, B-Iou)-Basketball-Court: nan
  IoU-Boeing737: 0.0000
  BoundaryIoU-Boeing737: 0.0000
  min(IoU, B-Iou)-Boeing737: 0.0000
  IoU-Boeing747: 9.6616
  BoundaryIoU-Boeing747: 0.0000
  min(IoU, B-Iou)-Boeing747: 0.0000
  IoU-Boeing777: 0.0000
  BoundaryIoU-Boeing777: 0.0000
  min(IoU, B-Iou)-Bo